# NB03 — Construcción y entrenamiento de una CNN
**Correspondencia: Semanas 9–10**


## 1. Preparación del entorno


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


## 2. Conectar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Definir el dataset

Utilice el dataset preparado durante el NB02.

La estructura esperada es:

```text
dataset/
├── clase_1/
├── clase_2/
├── clase_3/
└── clase_4/
```


In [ ]:
DATASET_DIR = "/content/drive/MyDrive/MachineLearning2026/dataset"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


## 4. Crear conjuntos de entrenamiento y validación


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Clases:", class_names)
print("Número de clases:", NUM_CLASSES)


## 5. Visualizar imágenes del dataset


In [ ]:
plt.figure(figsize=(9, 9))

for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 6. Optimizar el pipeline de datos


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


## 7. Data Augmentation

Las transformaciones se aplicarán solamente durante el entrenamiento.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
], name="data_augmentation")


## 8. Construcción de la CNN

La arquitectura combinará:

**Convolución → activación → pooling**

para extraer progresivamente características de las imágenes.

Posteriormente, las características serán utilizadas por capas densas para realizar la clasificación.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


## 9. Analizar la arquitectura

Observe especialmente:

- dimensiones de entrada y salida;
- cantidad de filtros;
- reducción espacial después de cada pooling;
- cantidad de parámetros;
- capa de clasificación final.


In [ ]:
print("Parámetros totales:", model.count_params())


## 10. Compilar la CNN


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 11. Entrenamiento inicial


In [ ]:
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


## 12. Curvas de accuracy


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Entrenamiento")
plt.plot(history_df["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy durante el entrenamiento")
plt.legend()
plt.show()


## 13. Curvas de pérdida


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Entrenamiento")
plt.plot(history_df["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.show()


## 14. Evaluación sobre validación


In [ ]:
val_loss, val_accuracy = model.evaluate(
    val_ds,
    verbose=0
)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")


## 15. Realizar predicciones


In [ ]:
for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)

    predicted_labels = np.argmax(predictions, axis=1)
    confidence = np.max(predictions, axis=1)

    resultados = pd.DataFrame({
        "Clase_real": [class_names[int(x)] for x in labels.numpy()],
        "Clase_predicha": [class_names[int(x)] for x in predicted_labels],
        "Confianza": confidence
    })

resultados.head(10)


## 16. Visualizar predicciones


In [ ]:
for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)
    confidence = np.max(predictions, axis=1)

    plt.figure(figsize=(10, 10))

    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        real = class_names[int(labels[i])]
        pred = class_names[int(predicted_labels[i])]
        conf = confidence[i]

        plt.title(
            f"Real: {real}\nPred: {pred} ({conf:.1%})"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 17. Matriz de confusión


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    predicted = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Matriz de confusión")
plt.show()


## 18. Métricas por clase


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


## 19. Guardar el modelo

El modelo guardado podrá utilizarse posteriormente para continuar el proyecto.


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/MachineLearning2026/cnn_inicial.keras"

model.save(MODEL_PATH)

print("Modelo guardado en:")
print(MODEL_PATH)


## 20. Experimentación con la arquitectura

Construya una segunda CNN modificando **uno o dos elementos** de la arquitectura inicial.

Puede experimentar con:

- número de filtros;
- cantidad de bloques convolucionales;
- tamaño del kernel;
- Dropout;
- cantidad de neuronas densas;
- learning rate;
- número de épocas.

Evite modificar simultáneamente demasiados elementos, ya que dificultará interpretar qué cambio produjo el resultado.


In [ ]:
modelo_experimental = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

modelo_experimental.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo_experimental.summary()


## 21. Entrenar el modelo experimental


In [ ]:
history_exp = modelo_experimental.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

exp_loss, exp_accuracy = modelo_experimental.evaluate(
    val_ds,
    verbose=0
)

print(f"Validation accuracy modelo experimental: {exp_accuracy:.4f}")


## 22. Comparar modelos


In [ ]:
comparacion = pd.DataFrame({
    "Modelo": [
        "CNN inicial",
        "CNN experimental"
    ],
    "Validation_accuracy": [
        val_accuracy,
        exp_accuracy
    ],
    "Parametros": [
        model.count_params(),
        modelo_experimental.count_params()
    ]
})

comparacion


## 23. Actividad

A partir del dataset del proyecto integrador:

1. Construya una CNN funcional.
2. Identifique las capas convolucionales y de pooling.
3. Registre la cantidad total de parámetros.
4. Entrene el modelo y analice las curvas de accuracy y loss.
5. Identifique posibles señales de overfitting o underfitting.
6. Evalúe el comportamiento por clase.
7. Construya una segunda arquitectura.
8. Compare ambas CNN utilizando evidencia.
9. Determine cuál utilizaría como punto de partida para la siguiente etapa del proyecto.
10. Guarde el modelo y registre claramente su versión.


## 24. Base para la Evaluación 3

Este notebook inicia el desarrollo técnico de la **Evaluación 3**.

El resultado esperado es disponer de una primera CNN entrenada y evaluada sobre el dataset preparado en E2.

**Dataset preparado → arquitectura CNN → entrenamiento → validación → métricas → comparación → modelo CNN inicial**

En el NB04 se continuará con **Transfer Learning, Fine-Tuning y selección del modelo definitivo**.
